## Upload file to S3 bucket

In [ ]:
import boto3
import uuid
import os
from boto3.s3.transfer import TransferConfig

# =========================================================
# CONFIGURATION
# =========================================================

# AWS Region
REGION = "us-east-1"

# Local file path
FILE_PATH = "test_file.txt"

# Generate unique bucket name
bucket_name = f"test-agent-{uuid.uuid4().hex[:8]}"

# S3 object key (folder/file structure inside bucket)
s3_key = f"uploads/{os.path.basename(FILE_PATH)}"

# =========================================================
# CREATE AWS CLIENTS
# =========================================================

s3_client = boto3.client(
    "s3",
    region_name=REGION,
    verify=False   # remove if SSL works properly
)

# =========================================================
# CREATE S3 BUCKET
# =========================================================

try:

    print(f"\nCreating bucket: {bucket_name}")

    # us-east-1 has special create_bucket behavior
    if REGION == "us-east-1":

        s3_client.create_bucket(
            Bucket=bucket_name
        )

    else:

        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={
                "LocationConstraint": REGION
            }
        )

    print("✅ Bucket created successfully")

except Exception as e:

    print("❌ Bucket creation failed")
    print(str(e))

# =========================================================
# UPLOAD CONFIG (MULTIPART FOR LARGE FILES)
# =========================================================

config = TransferConfig(
    multipart_threshold=1024 * 25,
    max_concurrency=10,
    multipart_chunksize=1024 * 25,
    use_threads=True
)

# =========================================================
# PROGRESS TRACKER
# =========================================================

class ProgressPercentage:
    def __init__(self, filename):
        self._filename = filename
        self._size = float(os.path.getsize(filename))
        self._seen_so_far = 0

    def __call__(self, bytes_amount):
        self._seen_so_far += bytes_amount
        percentage = (self._seen_so_far / self._size) * 100

        print(
            f"\rUploading... {percentage:.2f}% complete",
            end=""
        )

# =========================================================
# UPLOAD FILE TO S3
# =========================================================

try:

    print(f"\nUploading file: {FILE_PATH}")

    s3_client.upload_file(
        Filename=FILE_PATH,
        Bucket=bucket_name,
        Key=s3_key,
        Config=config,
        Callback=ProgressPercentage(FILE_PATH)
    )

    print("\n✅ Upload completed successfully!")

    print("\nS3 File Location:")
    print(f"s3://{bucket_name}/{s3_key}")

except Exception as e:

    print("\n❌ Upload failed")
    print(str(e))

# =========================================================
# VERIFY FILE EXISTS
# =========================================================

try:

    response = s3_client.list_objects_v2(
        Bucket=bucket_name
    )

    print("\n📂 Bucket Contents:")

    for obj in response.get("Contents", []):
        print(f" - {obj['Key']}")

except Exception as e:

    print("❌ Failed to list bucket contents")
    print(str(e))